## Hyper-Personalized Marketing: A Journey with Grok

You’ve dodged the internet’s ad barrage one too many times—pop-ups, banners, that 5-second YouTube teaser you’d trade anything to skip. How many ads hit you daily? How many feel like they get you, not some generic nobody? If you’re a brand, how often do your regulars see offers that truly spark their interest? Spoiler: not nearly enough.

Imagine ads that hook strangers in their native tongue and campaigns that captivate your loyal fans with perfect precision. Large language models like xAI’s Grok make this possible. Once, it was impractical for marketing teams to manually sift through user profiles and craft bespoke messages, too slow, too costly. Now, feed Grok the right context, and it takes over, blending nuanced language mastery with image-generation prowess to deliver hyper-personalized content across languages, formats, and tones in mere seconds.

## Table of Contents
- [Crafting Customers: Grok Builds Your Audience Profiles](#crafting-customers-grok-builds-your-audience-profiles)
  - [Synthetic Customer Generation](#synthetic-customer-generation)
  - [Persona Generation](#persona-generation)
- [Hyper Personalized Prompt](#hyper-personalized-prompt)
- [Image Generation](#image-generation)
  - [Meta-Prompting](#meta-prompting)
- [Final Results](#final-results)
- [Conclusion](#conclusion)

In [1]:
%pip install --quiet xai-sdk python-dotenv tqdm pandas aiofiles rich ipython

Note: you may need to restart the kernel to use updated packages.


### Crafting Customers: Grok Builds Your Audience Profiles

Every campaign needs a target. For ads, it’s new users with basic traits. For marketing, it’s your known crowd, rich with history. Typically, you’d pull from your database, but here, we'll use Grok to create synthetic sample profiles from scratch. First, we'll use the structured outputs feature to create sample customers with attributes that would aid in generating targeted marketing/ad content. Second, we’ll task Grok with fleshing out vivid, concrete personas—written snapshots that bring these profiles to life, ready to inspire hyper-personalized campaigns.

#### Synthetic Customer Generation

Below, we define a Pydantic model to shape sample customers with attributes typical of an e-commerce setting. Then, we harness Grok’s structured outputs feature to whip up 10 diverse profiles in one go.

In [2]:
import os

from dotenv import load_dotenv
from xai_sdk import AsyncClient
from xai_sdk.chat import user, system

from rich import print_json

load_dotenv()

GROK_IMAGINE_IMAGES = "grok-imagine-image"

GROK_CHAT_MODEL = "grok-4.3"

client = AsyncClient(
    api_key=os.getenv("XAI_API_KEY")
)
print("Status: Okay!")

Status: Okay!


In [3]:
from enum import Enum

from pydantic import BaseModel


class Gender(Enum):
    MALE = "MALE"
    FEMALE = "FEMALE"


class Item(BaseModel):
    name: str
    category: str
    price: float
    purchase_date: str


class Customer(BaseModel):
    name: str
    age: int
    location: str
    gender: Gender
    language: str
    purchase_history: list[Item]
    interests: list[str]
    search_history: dict[str, str]
    preferred_device: str
    persona: str | None = None


class Customers(BaseModel):
    customers: list[Customer]

In [4]:
async def generate_customers(
    client: AsyncClient, num_customers: int = 10, model: str = GROK_CHAT_MODEL
) -> Customers:
    prompt = f"""
    Generate {num_customers} sample customers with the following attributes:
    - age: int
    - location: str
    - gender: Gender
    - language: str
    - purchase_history: list[Item]
    - interests: list[str]
    - search_history: dict[str, str]
    - preferred_device: str

    Here are the attributes of an Item:
    - name: str
    - category: str
    - price: float
    - purchase_date: str

    Each customer generated should varied and distinct from the others:
    - ensure a variety of languages are represented and not just english, however try to focus on commonly spoken languages instead of niche ones.
    - ensure the customers are from a variety of different countries and not just from one country
    - ensure the language attribute is set to the most commonly spoken language in that country
    - Have more english speaking customers than non-english speaking ones.

    Please set the persona attribute to null for all customers.
    """

    chat = client.chat.create(
        model=model,
        messages=[system(prompt)],
        response_format=Customers,
        temperature=1.2,
    )

    _, customers = await chat.parse(Customers)

    if not customers.customers:
        raise ValueError("No customers generated!")

    return customers

In [5]:
customers = await generate_customers(
    client,
    num_customers=10,
    model=GROK_CHAT_MODEL
)

In [6]:
for customer in customers.customers:
    # use print_json for colorful output
    print_json(customer.model_dump_json(indent=2))

{
  "name": "John Smith",
  "age": 35,
  "location": "United States",
  "gender": "MALE",
  "language": "English",
  "purchase_history": [
    {
      "name": "Laptop",
      "category": "Electronics",
      "price": 999.99,
      "purchase_date": "2023-11-20"
    }
  ],
  "interests": [
    "technology",
    "hiking"
  ],
  "search_history": {
    "best wireless headphones": "electronics"
  },
  "preferred_device": "laptop",
  "persona": null
}

{
  "name": "Emma Thompson",
  "age": 28,
  "location": "United Kingdom",
  "gender": "FEMALE",
  "language": "English",
  "purchase_history": [
    {
      "name": "Novel",
      "category": "Books",
      "price": 14.99,
      "purchase_date": "2024-02-05"
    }
  ],
  "interests": [
    "reading",
    "yoga"
  ],
  "search_history": {
    "romance novels": "books"
  },
  "preferred_device": "tablet",
  "persona": null
}

{
  "name": "Liam Wilson",
  "age": 42,
  "location": "Australia",
  "gender": "MALE",
  "language": "English",
  "purchase_history": [
    {
      "name": "Grill Set",
      "category": "Outdoor",
      "price": 129.5,
      "purchase_date": "2023-12-10"
    }
  ],
  "interests": [
    "barbecuing",
    "surfing"
  ],
  "search_history": {
    "bbq accessories": "home"
  },
  "preferred_device": "mobile",
  "persona": null
}

{
  "name": "Sophia Patel",
  "age": 31,
  "location": "Canada",
  "gender": "FEMALE",
  "language": "English",
  "purchase_history": [
    {
      "name": "Winter Coat",
      "category": "Fashion",
      "price": 249.0,
      "purchase_date": "2024-01-22"
    }
  ],
  "interests": [
    "skiing",
    "photography"
  ],
  "search_history": {
    "camera gear": "electronics"
  },
  "preferred_device": "desktop",
  "persona": null
}

{
  "name": "Conor Kelly",
  "age": 25,
  "location": "Ireland",
  "gender": "MALE",
  "language": "English",
  "purchase_history": [
    {
      "name": "Guitar",
      "category": "Music",
      "price": 399.99,
      "purchase_date": "2023-10-15"
    }
  ],
  "interests": [
    "music",
    "football"
  ],
  "search_history": {
    "indie rock albums": "entertainment"
  },
  "preferred_device": "mobile",
  "persona": null
}

{
  "name": "Olivia Brown",
  "age": 38,
  "location": "New Zealand",
  "gender": "FEMALE",
  "language": "English",
  "purchase_history": [
    {
      "name": "Hiking Boots",
      "category": "Outdoor",
      "price": 89.95,
      "purchase_date": "2024-03-01"
    }
  ],
  "interests": [
    "hiking",
    "wine tasting"
  ],
  "search_history": {
    "best trails": "travel"
  },
  "preferred_device": "tablet",
  "persona": null
}

{
  "name": "Miguel Hernandez",
  "age": 45,
  "location": "Mexico",
  "gender": "MALE",
  "language": "Spanish",
  "purchase_history": [
    {
      "name": "Tennis Racket",
      "category": "Sports",
      "price": 79.99,
      "purchase_date": "2023-09-30"
    }
  ],
  "interests": [
    "tennis",
    "cooking"
  ],
  "search_history": {
    "mexican recipes": "food"
  },
  "preferred_device": "mobile",
  "persona": null
}

{
  "name": "Li Wei",
  "age": 33,
  "location": "China",
  "gender": "FEMALE",
  "language": "Mandarin",
  "purchase_history": [
    {
      "name": "Smart Watch",
      "category": "Electronics",
      "price": 199.0,
      "purchase_date": "2024-01-08"
    }
  ],
  "interests": [
    "fitness",
    "travel"
  ],
  "search_history": {
    "fitness trackers": "electronics"
  },
  "preferred_device": "mobile",
  "persona": null
}

{
  "name": "Marie Dubois",
  "age": 29,
  "location": "France",
  "gender": "FEMALE",
  "language": "French",
  "purchase_history": [
    {
      "name": "Perfume Set",
      "category": "Beauty",
      "price": 65.5,
      "purchase_date": "2023-12-25"
    }
  ],
  "interests": [
    "fashion",
    "art"
  ],
  "search_history": {
    "luxury cosmetics": "beauty"
  },
  "preferred_device": "desktop",
  "persona": null
}

{
  "name": "Hans Mueller",
  "age": 40,
  "location": "Germany",
  "gender": "MALE",
  "language": "German",
  "purchase_history": [
    {
      "name": "Power Drill",
      "category": "Tools",
      "price": 149.99,
      "purchase_date": "2024-02-18"
    }
  ],
  "interests": [
    "DIY",
    "cars"
  ],
  "search_history": {
    "tool reviews": "home"
  },
  "preferred_device": "laptop",
  "persona": null
}

#### Persona Generation

With our synthetic customers in hand, we can enrich them with detailed personas—textual summaries that make them feel real. Using the `AsyncClient`, we generate personas for all 10 customers concurrently.

In [7]:
from tqdm.asyncio import tqdm_asyncio


async def generate_persona(
    client: AsyncClient, customer: Customer, model: str = GROK_CHAT_MODEL
) -> str:
    prompt = f"""
    Generate a rich, 1-2 sentence persona for the following customer: {customer.model_dump_json(indent=2)}. The persona should suitable such that it can be used to generate a hyper-personalized ad or marketing piece.
    """

    chat = client.chat.create(model=model)
    chat.append(user(prompt))

    response = await chat.sample()

    persona = response.content

    if not persona:
        raise ValueError("No persona generated!")

    return persona


async def set_personas(
    client: AsyncClient, customers: list[Customer]
) -> list[Customer]:
    coroutines = [
        generate_persona(client, customer)
        for customer in customers
    ]

    personas = await tqdm_asyncio.gather(
        *coroutines,
        desc="Generating personas"
    )

    for customer, persona in zip(customers, personas):
        customer.persona = persona

    return customers

In [8]:
updated_customers = await set_personas(client, customers.customers)

I0515 21:04:37.968909 14138082 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0515 21:04:37.972308 14138849 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(81, generation: 1)
Generating personas: 100%|██████████████████████████████████████████████████████████████████| 10/10 [01:28<00:00,  8.85s/it]


In [9]:
for customer in updated_customers:
    print(customer.persona)

A 35-year-old American tech enthusiast and avid hiker who recently upgraded to a premium laptop and is actively researching top-tier wireless headphones to seamlessly blend his on-the-go productivity needs with immersive outdoor adventures. He values high-performance gear that supports both his professional efficiency and trail-focused lifestyle.
Emma is a mindful 28-year-old UK woman who finds daily balance through yoga and escapes into heartfelt romance novels on her tablet, craving stories that blend emotional depth with feel-good endings after her wellness routines.
Liam is a 42-year-old coastal Australian who splits his weekends between early-morning surf sessions and hosting laid-back backyard barbecues for family and friends. Recently having invested in a premium grill set, he’s now actively browsing for accessories on his mobile to level-up his outdoor cooking game and create memorable gatherings.
Sophia Patel is a 31-year-old Canadian adventurer and visual storyteller who thri

### Hyper Personalized Prompt

With customers and personas ready, it’s time to craft hyper-personalized marketing messages. The prompt below instructs Grok to generate short, punchy ads tailored to each customer’s unique profile, think 50-100 words that pop with excitement, weave in at least three specific attributes (like age, interests, or purchase history), and match their language and local vernacular. These messages pitch a fresh product tied to their data, complete with a compelling hook and a clear call-to-action.

The prompt below includes a number of techniques that help get the best out of Grok these include:
- Role assignment
- A well-defined task with specific criteria
- Detailed step-by-step instructions on how to perform the task
- Constraints to tell Grok what to avoid
- A reminder towards of the end of the prompt of the most important high-level instructions

In [10]:
async def generate_marketing_piece(
    client: AsyncClient, customer: Customer, model: str = GROK_CHAT_MODEL
) -> str:
    prompt = f"""
    You are an expert marketing content generator tasked with creating hyper-personalized, engaging, and exciting advertising content for individual customers. Your goal is to leverage detailed customer data to craft tailored messages that resonate with their demographics, behaviors, and preferences, while optionally aligning with a brand style guide where possible.

    Below is the Customer schema outlining the attributes that will be provided as input:
    {Customer.model_json_schema()}

    Task
    Given a single `Customer` instance (provided as input), generate a short, personalized marketing message (50-100 words) that:
    1. Feels exciting, urgent, or exclusive to grab attention.
    2. Incorporates at least 3 specific attributes from the customer's data (e.g., `age`, `purchase_history`, `interests`).
    3. Matches the customer's `language` and uses idiomatic lingo based on the customer's `location`.
    4. Aligns with their `preferred_device` for delivery.
    5. Suggests a specific product or service (either dynamically generated or inferred from data) tied to their purchase history, interests, or search history - but not something they have recently purchased.

    Instructions
    1. Analyze the Customer: Use attributes like `age`, `gender`, `location`, `purchase_history`, `interests`, `search_history`, and `preferred_device` to understand the customer. If `persona` is missing, infer a simple persona (e.g., "Tech-Savvy Gamer," "Eco-Chic Shopper") based on patterns.
    2. Generate a Product/Service (if not provided): If no specific product is given, create one based on the customer’s data. Example: For a customer with `interests: ["sustainability"]` and `purchase_history: ["Dress"]`, suggest a "Vegan Leather Bag" in the "Fashion" category for $45.
    2. Personalized Greeting: Always address the customer by their first name in the greeting
    3. Tailor the Tone: Adjust language and style to suit `age`, `gender`, and `interests`. For younger audiences, use casual, vibrant tones; for older audiences, use polished, professional tones.
    4. Use the language of the customer: Use idiomatic vernacular based on the customer's `location`.
    5. Use emojis: Use emojis to add a bit more flair to the message, but don't overdo it.
    6. Leverage Context: Reference `location` (e.g., local events), `purchase_date` (e.g., time since last purchase), or `search_history` (e.g., intent) for relevance.
    7. Create Hooks: Use storytelling ("Remember your last buy?"), exclusivity ("For our top shoppers only"), or FOMO ("This week only!") to engage.
    8. Personalize Offers: Suggest the product with a compelling deal (e.g., discount, bundle) tied to `purchase_history` or `search_history`. Use plausible pricing if generating a product.
    9. Optimize Delivery: Format for `preferred_device`—short and visual for "mobile," detailed for "desktop."
    10. Stay Concise: Keep the message 50-100 words, punchy, and action-oriented with a clear call-to-action (CTA), using new lines for clarity and structure where necessary.

    Constraints
    - Do not invent customer data not present in the `Customer` instance.
    - Avoid generic phrases like "Dear Customer" unless no personalization data is available.
    - Use the exact `language` specified (e.g., English, Japanese) and match cultural nuances where possible.
    - Do not exceed 100 words unless explicitly requested.

    Remember to:
    - Be creative but grounded in the data.
    - If generating a product, ensure it’s plausible and tied to the customer’s profile.
    - If unsure about cultural references, keep it simple and data-driven.
    - Only output the final advertising content and nothing else.

    Here is the customer:
    {customer.model_dump_json(indent=2, exclude={"persona"})}

    Here is the customer persona:
    {customer.persona}

    Here is the target language:
    {customer.language}
    """
    chat = client.chat.create(
        model=model,
        messages=[
            system(prompt)
        ],
    )

    response = await chat.sample()

    if not response.content:
        raise ValueError("No marketing piece generated")

    return response.content

In [11]:
marketing_pieces: list[str] = await tqdm_asyncio.gather(
    *[
        generate_marketing_piece(client, customer)
        for customer in updated_customers
    ]
)

100%|███████████████████████████████████████████████████████████████████████████████████████| 10/10 [01:07<00:00,  6.74s/it]


In [12]:
import pandas as pd

pd.set_option("display.max_colwidth", None)


df = pd.DataFrame(
    {
        "Customer JSON": [
            customer.model_dump_json(indent=2) for customer in updated_customers
        ],
        "Customer Persona": [customer.persona for customer in updated_customers],
        "Marketing Piece": marketing_pieces,
    }
)

df

,Customer JSON,Customer Persona,Marketing Piece
0,"{\n ""name"": ""John Smith"",\n ""age"": 35,\n ""location"": ""United States"",\n ""gender"": ""MALE"",\n ""language"": ""English"",\n ""purchase_history"": [\n {\n ""name"": ""Laptop"",\n ""category"": ""Electronics"",\n ""price"": 999.99,\n ""purchase_date"": ""2023-11-20""\n }\n ],\n ""interests"": [\n ""technology"",\n ""hiking""\n ],\n ""search_history"": {\n ""best wireless headphones"": ""electronics""\n },\n ""preferred_device"": ""laptop"",\n ""persona"": ""A 35-year-old American tech enthusiast and avid hiker who recently upgraded to a premium laptop and is actively researching top-tier wireless headphones to seamlessly blend his on-the-go productivity needs with immersive outdoor adventures. He values high-performance gear that supports both his professional efficiency and trail-focused lifestyle.""\n}",A 35-year-old American tech enthusiast and avid hiker who recently upgraded to a premium laptop and is actively researching top-tier wireless headphones to seamlessly blend his on-the-go productivity needs with immersive outdoor adventures. He values high-performance gear that supports both his professional efficiency and trail-focused lifestyle.,"Hey John, \n\nYour sleek laptop upgrade last fall was a game-changer—now pair it with our Apex Wireless Headphones. Built for tech-savvy hikers like you, they deliver noise-cancelling beats that crush trail wind and office buzz alike. \n\nGrab yours today with 20% off for top customers—offer ends this week. Ready to hike smarter?"
1,"{\n ""name"": ""Emma Thompson"",\n ""age"": 28,\n ""location"": ""United Kingdom"",\n ""gender"": ""FEMALE"",\n ""language"": ""English"",\n ""purchase_history"": [\n {\n ""name"": ""Novel"",\n ""category"": ""Books"",\n ""price"": 14.99,\n ""purchase_date"": ""2024-02-05""\n }\n ],\n ""interests"": [\n ""reading"",\n ""yoga""\n ],\n ""search_history"": {\n ""romance novels"": ""books""\n },\n ""preferred_device"": ""tablet"",\n ""persona"": ""Emma is a mindful 28-year-old UK woman who finds daily balance through yoga and escapes into heartfelt romance novels on her tablet, craving stories that blend emotional depth with feel-good endings after her wellness routines.""\n}","Emma is a mindful 28-year-old UK woman who finds daily balance through yoga and escapes into heartfelt romance novels on her tablet, craving stories that blend emotional depth with feel-good endings after her wellness routines.","Hi Emma,\n\nAfter your latest yoga flow and that heartfelt novel from February, treat yourself to our exclusive ""Bliss & Balance"" Romance Bundle—three mindful love stories perfect for tablet escapes. Just £22.99 (20% off this week only)! \n\nTap now and unwind in true UK style. 📖🧘‍♀️"
2,"{\n ""name"": ""Liam Wilson"",\n ""age"": 42,\n ""location"": ""Australia"",\n ""gender"": ""MALE"",\n ""language"": ""English"",\n ""purchase_history"": [\n {\n ""name"": ""Grill Set"",\n ""category"": ""Outdoor"",\n ""price"": 129.5,\n ""purchase_date"": ""2023-12-10""\n }\n ],\n ""interests"": [\n ""barbecuing"",\n ""surfing""\n ],\n ""search_history"": {\n ""bbq accessories"": ""home""\n },\n ""preferred_device"": ""mobile"",\n ""persona"": ""Liam is a 42-year-old coastal Australian who splits his weekends between early-morning surf sessions and hosting laid-back backyard barbecues for family and friends. Recently having invested in a premium grill set, he’s now actively browsing for accessories on his mobile to level-up his outdoor cooking game and create memorable gatherings.""\n}","Liam is a 42-year-old coastal Australian who splits his weekends between early-morning surf sessions and hosting laid-back backyard barbecues for family and friends. Recently having invested in a premium grill set, he’s now actively browsing for accessories on his mobile to level-up his outdoor cooking game and create memorable gatherings.","Hey Liam, \n\nReady to fire up the ultimate Aussie barbie? Since scoring that Grill Set l

In just a matter of seconds we've created hyper personalized relevant marketing material, in a variety of vastly different languages, with idiomatic language. Our prompt above focuses on short punchy ads but this can easily be adjusted to fit a specific format or style to suit your needs.

### Image Generation

Why stop at text? Let’s amp up these marketing pieces with eye-catching images. Using Grok’s [image-generation capabilities](https://docs.x.ai/docs/guides/image-generations), we’ll create visuals based on each marketing message and persona. These images spotlight the product in a stylized, engaging scene—think vibrant watercolors or bold graphics tied to the customer’s interests and lifestyle, all without humans to keep the focus tight.

#### Meta-Prompting

Typically, the image generation API works by taking a prompt that describes the image you want created. We could pass a prompt directly here, but instead, we use a meta-prompt, a prompt fed into a non-image-generation model to craft a tailored prompt optimized for the image gen API. This lets us get ultra-specific, ensuring the generated prompt perfectly aligns with the customer’s persona and marketing piece for standout visuals.

> Note: The image generation API actually takes the prompt you give it and [re-writes](https://docs.x.ai/docs/guides/image-generations#image-generations) it anyway, which might make this step seem redundant. However, even with that re-writing process, providing a high quality and descriptive initial prompt often leads to better quality end images.

In [13]:
async def generate_image_prompt(
    client: AsyncClient,
    marketing_piece: str,
    customer_persona: str,
    model: str = GROK_CHAT_MODEL,
) -> str:
    meta_prompt = f"""
    Create a concise and detailed prompt optimized for an image generation model based on the persona and marketing piece provided below.

    The generated prompt should:
    - Instruct the model to create an image promoting the product or service from the marketing piece, with the product or service as the central focus.
    - Exclude humans from the image.
    - Place the product or service in a visually engaging, stylized context that reflects the persona’s interests, lifestyle, or preferences.  
    - Include subtle background elements or details that tie into the persona’s characteristics (e.g., hobbies, environment) to enhance relevance, without distracting from the product or service.  
    - Use vivid, descriptive language to inspire a creative and unique visual output, keeping the tone imaginative rather than literal.
    - Instruct the model to use different styles, not all photos need to be hyper-realistic but some may be more artistic, sketched, paintings, watercolor style etc. The style used should be chosen based on the marketing piece and persona.
    - Format the output as a single, flowing sentence or short paragraph, avoiding lists or numbered steps, suitable for direct input into an image generation tool.

    Steps to Generate the Prompt:
    1. Analyze the marketing piece to pinpoint the product or service and its key selling points.
    2. Examine the persona to extract specific traits, interests, or motivations that connect to the product or service.
    3. Craft a scene where the product or service shines in a context meaningful to the persona, such as the city or location, blending in subtle, complementary details based on the persona's characteristics. 
    4. Ensure the language is evocative, visual, and concise, tailored for an image generation model’s interpretation.

    Persona:
    {customer_persona}

    Marketing Piece:
    {marketing_piece}
    """

    chat = client.chat.create(
        model=model,
        messages=[
            user(meta_prompt)
        ],
        max_tokens=400,
    )

    response = await chat.sample()

    if not response.content:
        raise ValueError("No image prompt generated!")

    return response.content

In [14]:
import base64
import os
import uuid

import aiofiles


async def generate_image(
    client: AsyncClient, marketing_piece: str, customer_persona: str, filepath: str
) -> str:
    image_prompt = await generate_image_prompt(
        client,
        marketing_piece,
        customer_persona
    )

    print(image_prompt)

    response = await client.image.sample(
        model=GROK_IMAGINE_IMAGES,
        prompt=image_prompt,
    )

    image_data = await response.image

    # create a folder if it doesn't exist
    os.makedirs(filepath, exist_ok=True)
    # generate a random file name
    filename = os.path.join(filepath, f"generated_image_{uuid.uuid4().hex[:5]}.png")

    async with aiofiles.open(filename, "wb") as f:
        await f.write(image_data)

    return filename

In [15]:
async def generate_all_images(client: AsyncClient, df: pd.DataFrame, images_dir: str):
    tasks = [
        generate_image(
            client, row["Marketing Piece"], row["Customer Persona"], images_dir
        )
        for _, row in df.iterrows()
    ]

    image_paths = await tqdm_asyncio.gather(*tasks)

    df["image_path"] = image_paths
    return df

In [ ]:
df = await generate_all_images(client, df, "generated_images/")

  0%|                                                                                                | 0/10 [00:00<?, ?it/s]

Create a vibrant stylized digital illustration of sleek Apex Wireless Headphones as the central focus, resting on a weathered granite outcrop amid a misty mountain trail at golden sunrise, with subtle glowing circuit patterns and soft wireless signal waves seamlessly merging into the pine forest canopy and distant peaks, capturing a dynamic fusion of premium tech innovation and immersive outdoor adventure in rich, imaginative colors and cinematic lighting.
Create a vibrant stylized digital illustration of a sleek premium portable blender as the central focus, positioned on a sun-drenched clay tennis court in Mexico with subtle background details like a racket resting nearby, scattered mangoes, limes, and chili peppers hinting at traditional recipes, all rendered in an energetic watercolor style with bold cultural patterns and dynamic lighting to evoke post-match refreshment and Mexican flair.
Create a vivid promotional image centered on the premium Trail & Vine Backpack resting on a mo

 10%|████████▊                                                                               | 1/10 [00:10<01:30, 10.04s/it]

Create a dreamy watercolor illustration of the "Bliss & Balance" Romance Bundle—three elegantly bound mindful love stories with soft pastel covers—resting invitingly on a folded yoga mat beside an open glowing tablet in a tranquil English cottage garden at dawn, surrounded by subtle dew-kissed roses, lavender sprigs, and gentle mist, all bathed in warm, serene light that evokes peaceful mindfulness and heartfelt romance, rendered in an artistic, flowing watercolor style with delicate details and a cozy UK atmosphere.


 70%|█████████████████████████████████████████████████████████████▌                          | 7/10 [00:13<00:03,  1.02s/it]

A vibrant stylized digital illustration centers the sleek black Portable Amp Pro on an old wooden guitar case, its glowing phone-connected interface pulsing with indie-rock waveforms, set against a dreamy fusion of misty Irish hills and a distant floodlit Premier League pitch at twilight, with faint floating vinyl records, guitar picks, and emerald-green football scarves woven subtly into the energetic background, rendered in bold pop-art strokes of electric blue and warm amber to capture spontaneous creativity and gig-ready freedom.


 80%|██████████████████████████████████████████████████████████████████████▍                 | 8/10 [00:18<00:04,  2.12s/it]

An exquisite limited-edition Art Couture makeup palette with vibrant contemporary-art-inspired shades rests as the radiant centerpiece on a sleek marble vanity in a sunlit Parisian atelier, surrounded by subtle flowing silk textiles and delicate abstract brushstroke motifs echoing modern masters, all rendered in an elegant watercolor-and-ink style that evokes luxurious wearable beauty rituals and refined French fashion sensibility.


 90%|███████████████████████████████████████████████████████████████████████████████▏        | 9/10 [00:25<00:03,  3.37s/it]

### Final Results

Now that we’ve generated images and personas, let’s take a look at how these hyper-personalized marketing pieces come together with their custom visuals to captivate each customer.

In [ ]:
from IPython.display import Image

for index, row in df.iterrows():
    print(f"{row['Marketing Piece']}\n")
    display(Image(row["image_path"], width=600))
    print()

Pretty cool, right?

## Conclusion

Starting with nothing, we’ve built a powerful pipeline that runs in the order of minutes to deliver hyper-personalized marketing messages along with custom images for each user. Let's quickly recap some of what we covered:

- Synthetic Data Generation using Structured Outputs
- Using prompt engineering best practices to generate hyper personalized marketing messages
- Using meta prompting to generate prompts optimized for image generation

We only scratched the surface, here are some ideas for you to try to take this even further:
- Update the text generation prompt to incorporate you're organization's brand voice/style guide
- Update the text generation prompt so that it can adhere to a specific template for the generated message.

